<a href="https://colab.research.google.com/github/shoh0806/Deep-Learning-Project/blob/main/Late_Fusion_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Late Fusion 멀티모달 클릭베이트 분류 모델

이 노트북은 **BERT 텍스트 인코더 + ViT 이미지 인코더 + Concatenation + MLP** 구조의 Late Fusion 베이스라인입니다.

- 텍스트: `bert-base-cased`
- 이미지: `google/vit-base-patch16-224`
- Fusion: BERT `[CLS]` 벡터와 ViT `[CLS]` 벡터를 단순 결합
- 분류기: `Linear(1536 → 256) → ReLU → Dropout → Linear(256 → 2)`
- 평가: Test Loss, Test Accuracy, Test F1-score, Classification Report, Confusion Matrix

기존 코드의 핵심 하이퍼파라미터와 재현성 설정은 유지했습니다.

## 1. 라이브러리 설치

In [ ]:
!pip install transformers -q

## 2. 라이브러리 import 및 재현성 설정

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertModel
from transformers import ViTModel, ViTImageProcessor

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ==============================
# 재현성 설정
# ==============================
SEED = 42

def set_seed(seed=42):
    # 가능한 한 같은 조건에서 같은 결과가 나오도록 seed를 고정한다.
    # 단, GPU/Colab 환경에서는 일부 연산 때문에 완전한 100% 재현이 어려울 수 있다.
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    # CuDNN 연산의 비결정성 감소
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # 일부 PyTorch 연산을 결정론적으로 실행
    # Colab 환경에서 특정 연산이 막히는 것을 피하기 위해 warn_only=True 사용
    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_worker(worker_id):
    # DataLoader worker별 seed 고정
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")
print(f"Seed: {SEED}")

## 3. Google Drive 연결 및 경로 설정

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ==============================
# 프로젝트 경로 설정
# ==============================
BASE_DIR = "/content/drive/MyDrive/clickbait_project"
CSV_PATH = f"{BASE_DIR}/dataset.csv"
IMG_DIR = f"{BASE_DIR}/thumbnails"

# 가장 좋은 validation F1-score를 기록한 모델 저장 경로
SAVE_PATH = f"{BASE_DIR}/late_fusion_best.pt"

print("CSV_PATH :", CSV_PATH)
print("IMG_DIR  :", IMG_DIR)
print("SAVE_PATH:", SAVE_PATH)

## 4. 데이터 로드 및 기본 확인

In [ ]:
df = pd.read_csv(CSV_PATH)

# 필요한 컬럼 확인
required_columns = {"video_id", "title", "label"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"dataset.csv에 필요한 컬럼이 없습니다: {missing_columns}")

# label 정수형 변환
df["label"] = df["label"].astype(int)

print(f"전체 데이터 수: {len(df)}개")
print(f"클릭베이트(1): {(df['label'] == 1).sum()}개")
print(f"정상(0): {(df['label'] == 0).sum()}개")
print()
print(df.head())

## 5. 데이터 분할

In [ ]:
# 기존 코드와 동일한 분할 방식 유지
# 전체 데이터의 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["label"]
)

print(f"Train      : {len(train_df)}개")
print(f"Validation : {len(val_df)}개")
print(f"Test       : {len(test_df)}개")


## 6. Tokenizer / Image Processor 준비

In [ ]:
# 기존 코드와 동일한 pretrained 모델 유지(mbert)
TEXT_MODEL_NAME = "bert-base-multilingual-cased"
IMAGE_MODEL_NAME = "google/vit-base-patch16-224"

tokenizer = BertTokenizer.from_pretrained(TEXT_MODEL_NAME)
image_processor = ViTImageProcessor.from_pretrained(IMAGE_MODEL_NAME)

print("Tokenizer 및 Image Processor 준비 완료")

## 7. Dataset 및 DataLoader 정의

In [ ]:
class MultimodalDataset(Dataset):
    # 유튜브 제목 텍스트와 썸네일 이미지를 함께 반환하는 Dataset
    def __init__(self, dataframe, tokenizer, image_processor, img_dir, max_len=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.img_dir = img_dir
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def _get_image_path(self, row):
        # 1순위: dataset.csv에 thumbnail_path 컬럼이 있으면 그 경로 사용
        # 2순위: 없으면 기존 코드처럼 IMG_DIR/video_id.jpg 사용
        if "thumbnail_path" in row.index and pd.notna(row["thumbnail_path"]):
            path = str(row["thumbnail_path"])

            if os.path.isabs(path):
                return path

            return os.path.join(BASE_DIR, path)

        return os.path.join(self.img_dir, f"{row['video_id']}.jpg")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        title = str(row["title"])
        label = int(row["label"])

        # ==============================
        # 텍스트 전처리: BERT 입력 생성
        # ==============================
        text_inputs = self.tokenizer(
            title,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # ==============================
        # 이미지 전처리: ViT 입력 생성
        # ==============================
        image_path = self._get_image_path(row)

        try:
            image = Image.open(image_path).convert("RGB")
        except FileNotFoundError:
            raise FileNotFoundError(f"이미지 파일을 찾을 수 없습니다: {image_path}")

        image_inputs = self.image_processor(
            images=image,
            return_tensors="pt"
        )

        return {
            "input_ids": text_inputs["input_ids"].squeeze(0),
            "attention_mask": text_inputs["attention_mask"].squeeze(0),
            "pixel_values": image_inputs["pixel_values"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

# ==============================
# 하이퍼파라미터 유지
# ==============================
MAX_LEN = 128
BATCH_SIZE = 16

train_dataset = MultimodalDataset(train_df, tokenizer, image_processor, IMG_DIR, max_len=MAX_LEN)
val_dataset = MultimodalDataset(val_df, tokenizer, image_processor, IMG_DIR, max_len=MAX_LEN)
test_dataset = MultimodalDataset(test_df, tokenizer, image_processor, IMG_DIR, max_len=MAX_LEN)

# DataLoader shuffle까지 재현성을 맞추기 위한 generator
g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    worker_init_fn=seed_worker,
    generator=g,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Dataset 및 DataLoader 준비 완료")
print(f"Train batch 수: {len(train_loader)}")
print(f"Validation batch 수: {len(val_loader)}")
print(f"Test batch 수: {len(test_loader)}")

## 8. Late Fusion 모델 정의 (bert, vit 마지막 2개 layer만 오픈)

In [ ]:
def freeze_backbone_except_last_layers(model, bert_unfreeze_layers=2, vit_unfreeze_blocks=2):
    """
    BERT와 ViT backbone은 대부분 freeze하고,
    마지막 일부 layer/block만 fine-tuning하도록 설정하는 함수
    """

    # ==============================
    # 1. BERT 전체 freeze
    # ==============================
    for param in model.bert.parameters():
        param.requires_grad = False

    # BERT 마지막 N개 encoder layer만 unfreeze
    for layer in model.bert.encoder.layer[-bert_unfreeze_layers:]:
        for param in layer.parameters():
            param.requires_grad = True

    # ==============================
    # 2. ViT 전체 freeze
    # ==============================
    for param in model.vit.parameters():
        param.requires_grad = False

    # ViT 마지막 N개 encoder block만 unfreeze
    for layer in model.vit.encoder.layer[-vit_unfreeze_blocks:]:
        for param in layer.parameters():
            param.requires_grad = True

    # ViT는 encoder 뒤에 layernorm이 있으므로 같이 열어주는 것이 좋음
    if hasattr(model.vit, "layernorm"):
        for param in model.vit.layernorm.parameters():
            param.requires_grad = True

In [ ]:
class LateFusionClassifier(nn.Module):
    # BERT와 ViT의 [CLS] 벡터를 단순 결합하는 Late Fusion 모델
    def __init__(self, hidden_dim=256, dropout_rate=0.3):
        super().__init__()

        self.bert = BertModel.from_pretrained(TEXT_MODEL_NAME)
        self.vit = ViTModel.from_pretrained(IMAGE_MODEL_NAME)

        # BERT와 ViT는 마지막 2개 layer/block만 fine-tuning
        freeze_backbone_except_last_layers(
            self,
            bert_unfreeze_layers=2,
            vit_unfreeze_blocks=2
        )

        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(768 + 768, hidden_dim)
        self.relu = nn.ReLU()
        self.classifier = nn.Linear(hidden_dim, 2)

    def forward(self, input_ids, attention_mask, pixel_values):
        # ==============================
        # 텍스트 인코딩
        # ==============================
        bert_outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_cls = bert_outputs.last_hidden_state[:, 0, :]  # (batch_size, 768)

        # ==============================
        # 이미지 인코딩
        # ==============================
        vit_outputs = self.vit(pixel_values=pixel_values)
        image_cls = vit_outputs.last_hidden_state[:, 0, :]  # (batch_size, 768)

        # ==============================
        # Late Fusion: 단순 Concatenation
        # ==============================
        fused = torch.cat([text_cls, image_cls], dim=1)  # (batch_size, 1536)

        # ==============================
        # MLP 분류기
        # ==============================
        x = self.dropout(fused)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.classifier(x)

        return logits

# 기존 하이퍼파라미터 유지
HIDDEN_DIM = 256
DROPOUT_RATE = 0.3

model = LateFusionClassifier(
    hidden_dim=HIDDEN_DIM,
    dropout_rate=DROPOUT_RATE
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Late Fusion 모델 준비 완료")
print(f"전체 파라미터 수: {total_params:,}")
print(f"학습 가능 파라미터 수: {trainable_params:,}")

## 9. Optimizer / Loss 설정

In [ ]:
# 기존 하이퍼파라미터 유지
LEARNING_RATE = 3e-5

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE
    )
criterion = nn.CrossEntropyLoss()

print("Optimizer 및 Loss 준비 완료")
print(f"Learning rate: {LEARNING_RATE}")

## 10. 학습 및 평가 함수 정의

In [ ]:
def compute_metrics(targets, preds):
    # Accuracy와 F1-score를 계산한다.
    # F1-score는 classification_report의 macro avg와 맞추기 위해 macro average를 사용한다.
    accuracy = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average="macro")

    return {
        "accuracy": accuracy,
        "f1_score": f1
    }

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    # 1 epoch 학습 함수
    model.train()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().tolist())
        all_targets.extend(labels.detach().cpu().tolist())

    avg_loss = total_loss / len(dataloader)
    metrics = compute_metrics(all_targets, all_preds)

    return avg_loss, metrics["accuracy"], metrics["f1_score"]

def evaluate(model, dataloader, criterion, device, return_predictions=False):
    # Validation/Test 평가 함수
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["label"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values
            )

            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.detach().cpu().tolist())
            all_targets.extend(labels.detach().cpu().tolist())

    avg_loss = total_loss / len(dataloader)
    metrics = compute_metrics(all_targets, all_preds)

    if return_predictions:
        return avg_loss, metrics["accuracy"], metrics["f1_score"], all_preds, all_targets

    return avg_loss, metrics["accuracy"], metrics["f1_score"]

print("학습 및 평가 함수 준비 완료")

## 11. 모델 학습

In [ ]:
# 기존 하이퍼파라미터 유지
EPOCHS = 10
PATIENCE = 3

best_val_f1 = 0.0
patience_counter = 0

history = {
    "train_loss": [],
    "train_accuracy": [],
    "train_f1_score": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_f1_score": []
}

print("========== Late Fusion Training Start ==========")

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy, train_f1 = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device
    )

    val_loss, val_accuracy, val_f1 = evaluate(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["train_f1_score"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)
    history["val_f1_score"].append(val_f1)

    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"  Train Loss      : {train_loss:.4f}")
    print(f"  Train Accuracy  : {train_accuracy:.4f}")
    print(f"  Train F1-score  : {train_f1:.4f}")
    print(f"  Val Loss        : {val_loss:.4f}")
    print(f"  Val Accuracy    : {val_accuracy:.4f}")
    print(f"  Val F1-score    : {val_f1:.4f}")

    # Validation F1-score 기준으로 best model 저장
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0

        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  → Best model 저장 완료: {SAVE_PATH}")
    else:
        patience_counter += 1
        print(f"  → Early stopping patience: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping 실행. 최고 Validation F1-score: {best_val_f1:.4f}")
            break

print("\n========== Late Fusion Training Finished ==========")
print(f"Best Validation F1-score: {best_val_f1:.4f}")

## 12. 학습 과정 확인

In [ ]:
history_df = pd.DataFrame(history)
history_df.index = history_df.index + 1
history_df.index.name = "epoch"

display(history_df)

## 13. 테스트 평가

In [ ]:
# 저장된 best model을 불러와서 테스트한다.
# SAVE_PATH가 위에서 정의되어 있으므로 NameError가 나지 않는다.
if os.path.exists(SAVE_PATH):
    model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
    print(f"저장된 best model을 불러왔습니다: {SAVE_PATH}")
else:
    print("저장된 best model이 없습니다. 현재 메모리에 있는 모델로 평가합니다.")

test_loss, test_accuracy, test_f1, test_preds, test_targets = evaluate(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=device,
    return_predictions=True
)

print("\n========== Late Fusion Test Result ==========")
print(f"Test Loss      : {test_loss:.4f}")
print(f"Test Accuracy  : {test_accuracy:.4f}")
print(f"Test F1-score  : {test_f1:.4f}")

print("\n[Classification Report]")
print(
    classification_report(
        test_targets,
        test_preds,
        target_names=["Non-Clickbait (0)", "Clickbait (1)"],
        digits=4
    )
)

print("[Confusion Matrix]")
cm = confusion_matrix(test_targets, test_preds)
print(cm)

## 14. Confusion Matrix 해석용 표

In [ ]:
cm_df = pd.DataFrame(
    cm,
    index=["Actual Non-Clickbait (0)", "Actual Clickbait (1)"],
    columns=["Predicted Non-Clickbait (0)", "Predicted Clickbait (1)"]
)

display(cm_df)

tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix 해석")
print(f"TN: 실제 정상(0)을 정상(0)으로 맞춘 개수        = {tn}")
print(f"FP: 실제 정상(0)을 클릭베이트(1)로 잘못 예측한 개수 = {fp}")
print(f"FN: 실제 클릭베이트(1)를 정상(0)으로 잘못 예측한 개수 = {fn}")
print(f"TP: 실제 클릭베이트(1)를 클릭베이트(1)로 맞춘 개수    = {tp}")